# AHR-MalCL temporal EMBER pilot

Standalone upstream EMBER 2018 v2 benchmark. The stream is **chronological new-to-learner family admission**; it does not claim that a family's first appearance in pilot training data is its real-world emergence date. January's initial top 50 come from January training-partition support only.

The production matrix has exactly nine runs: Full Hybrid, Anchor-only, and Current-only, each with seeds 42, 43, and 44. The package and results use a separate Drive root. **Training stays disabled until you explicitly change the two opt-in values in Run/Resume.**


## 1. Setup

Put `temporal_ember_pilot.zip` and its `.sha256` sidecar in `MyDrive/AHR-MalCL-temporal-ember-pilot/`. Open this notebook in Colab and run this cell. Setup mounts Drive, verifies the ZIP, extracts only the standalone package, and installs the pinned feature-vectorization libraries. Choose a GPU runtime before production training.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
from google.colab import drive

drive.mount("/content/drive")
PILOT_HOME = Path("/content/drive/MyDrive/AHR-MalCL-temporal-ember-pilot")
PILOT_HOME.mkdir(parents=True, exist_ok=True)
PACKAGE_ZIP = PILOT_HOME / "temporal_ember_pilot.zip"
PACKAGE_SHA_FILE = Path(str(PACKAGE_ZIP) + ".sha256")
if not PACKAGE_ZIP.exists() or not PACKAGE_SHA_FILE.exists():
    raise FileNotFoundError("Upload temporal_ember_pilot.zip and temporal_ember_pilot.zip.sha256 to MyDrive/AHR-MalCL-temporal-ember-pilot/")
expected_zip_sha = PACKAGE_SHA_FILE.read_text(encoding="ascii").split()[0].lower()
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()
actual_zip_sha = sha256_file(PACKAGE_ZIP)
if actual_zip_sha != expected_zip_sha:
    raise ValueError(f"Pilot ZIP checksum mismatch: expected {expected_zip_sha}, found {actual_zip_sha}")
EXTRACT_ROOT = Path("/content/temporal_ember_pilot_source")
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(PACKAGE_ZIP) as archive:
    root = EXTRACT_ROOT.resolve()
    for member in archive.infolist():
        destination = (EXTRACT_ROOT / member.filename).resolve()
        if not destination.is_relative_to(root):
            raise ValueError(f"Unsafe path in pilot ZIP: {member.filename}")
    archive.extractall(EXTRACT_ROOT)
PACKAGE_DIR = EXTRACT_ROOT / "temporal_ember_pilot"
if not (PACKAGE_DIR / "runner.py").exists():
    raise FileNotFoundError("The ZIP does not contain temporal_ember_pilot/runner.py")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "-r", str(PACKAGE_DIR / "requirements.txt")])
sys.path.insert(0, str(EXTRACT_ROOT))
import numpy as np
import torch
import sklearn
import lief
from temporal_ember_pilot.config import EXPERIMENT_ID, DEFAULT_OUTPUT_ROOT, validate_run_matrix
PILOT_EXPERIMENT_ID = "ahrmalcl_temporal_ember18v2_newfamily_v1"
if EXPERIMENT_ID != PILOT_EXPERIMENT_ID:
    raise ValueError("The package experiment ID differs from this notebook handoff")
from temporal_ember_pilot.data import load_manifest

OUTPUT_ROOT = PILOT_HOME / "outputs" / EXPERIMENT_ID
AUDIT_DIR = OUTPUT_ROOT / "audit"
REPORT_DIR = OUTPUT_ROOT / "report"
ARCHIVE_PATH = PILOT_HOME / "data" / "ember_dataset_2018_2.tar.bz2"
FEATURE_STORE_PATH = Path("/content/temporal_ember_pilot_feature_store")  # local memmap for faster randomized reads; set to OUTPUT_ROOT / "feature_store" to persist it in Drive
FROZEN_MANIFEST_PATH = PACKAGE_DIR / "split_manifest.json.gz"
STREAM_MODE = "chronological_new_to_learner_family_admission"
if STREAM_MODE != "chronological_new_to_learner_family_admission":
    raise ValueError("This package supports only the frozen chronological new-to-learner family stream")

manifest = load_manifest(FROZEN_MANIFEST_PATH)
schedule = json.loads((PACKAGE_DIR / "frozen_budget_schedule.json").read_text(encoding="utf-8"))
assert len(validate_run_matrix()) == 9
print(f"Verified package SHA-256: {actual_zip_sha}")
print(f"Experiment: {EXPERIMENT_ID}")
print(f"Frozen split SHA-256: {manifest['manifest_sha256']}")
print(f"Frozen update schedule SHA-256: {schedule['schedule_sha256']}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Progress file: {OUTPUT_ROOT / 'progress.json'}")
print(f"Checkpoint root: {OUTPUT_ROOT / 'runs'}")
print(f"Python={sys.version.split()[0]} NumPy={np.__version__} Torch={torch.__version__} sklearn={sklearn.__version__} LIEF={lief.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
for row in schedule["windows"]:
    print(f"{row['window_id']}: current={row['current_train_rows']}, planned anchors={row['planned_anchor_rows']}, planned generated={row['planned_generated_rows']}, classifier updates={row['classifier_updates_all_arms']}")


## 2. Audit

The audit reads only the six official `train_features_*.jsonl` members. It checks the published archive SHA-256, records the meaning and coverage of `appeared`, enforces the duplicate policy, and writes the audit report to this pilot's Drive root. It does not use the official test member. The feature extractor uses official EMBER v2 preprocessing; the published CoLLAs preprocessing is a reference only, and its randomized split is not reused.


In [ ]:
from temporal_ember_pilot.audit import (
    OFFICIAL_ARCHIVE_NAME, OFFICIAL_ARCHIVE_SHA256, audit_archive, sha256_file as audit_sha256_file,
)
from temporal_ember_pilot.data import load_manifest

ARCHIVE_URL = "https://ember.elastic.co/ember_dataset_2018_2.tar.bz2"
ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ARCHIVE_PATH.exists():
    temporary = ARCHIVE_PATH.with_suffix(ARCHIVE_PATH.suffix + ".partial")
    request = urllib.request.Request(ARCHIVE_URL, headers={"User-Agent": "AHR-MalCL-temporal-pilot/1.0"})
    downloaded = 0
    with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as target:
        while True:
            block = response.read(8 << 20)
            if not block:
                break
            target.write(block)
            downloaded += len(block)
            if downloaded and downloaded % (256 << 20) < (8 << 20):
                print(f"Downloaded {downloaded / (1024**3):.2f} GiB")
    os.replace(temporary, ARCHIVE_PATH)
if audit_sha256_file(ARCHIVE_PATH) != OFFICIAL_ARCHIVE_SHA256:
    raise ValueError("The downloaded archive does not match the published EMBER 2018 v2 SHA-256")
AUDIT_LOCAL_DIR = Path("/content/temporal_ember_pilot_audit")
AUDIT_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
def audit_progress(rows, members):
    print(f"Audited {rows:,} source rows across {len(members)} train shards")

audit_result = audit_archive(ARCHIVE_PATH, AUDIT_LOCAL_DIR, progress=audit_progress)
audited_manifest = load_manifest(AUDIT_LOCAL_DIR / "split_manifest.json.gz")
if audit_result["audit_status"] != "passed":
    raise ValueError("CPU data audit did not pass")
if audited_manifest["manifest_sha256"] != manifest["manifest_sha256"]:
    raise ValueError("The source archive audit differs from the release-frozen split manifest")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
for name in ("audit_report.json", "split_manifest.json.gz", "split_manifest.json.gz.sha256"):
    shutil.copy2(AUDIT_LOCAL_DIR / name, AUDIT_DIR / name)
AUDIT_READY = True
print(f"Audit passed; archive SHA-256: {audit_result['source_sha256']}")
print(f"Date field: {audit_result['date_field']['name']} — {audit_result['date_field']['meaning']}")
print(f"Date coverage: {audit_result['date_coverage']['minimum_month']} to {audit_result['date_coverage']['maximum_month']} ({audit_result['date_coverage']['distinct_months']} distinct months)")
print(f"Duplicate policy: {json.dumps(audit_result['duplicate_policy'], sort_keys=True)}")
for window, support in audit_result["family_support_by_window"].items():
    print(f"{window}: family-labeled={support['family_labeled_malware']:,}, families={support['unique_families']:,}, train={support['train_rows']:,}, heldout={support['heldout_rows']:,}")
print("Admission note: new-to-learner family admission is not a real-world emergence-date claim; January's initial top 50 are selected from January training rows only.")


## 3. Run/Resume

Production training is **disabled by default**. To start, explicitly set `ENABLE_TRAINING = True`, enter the exact experiment ID in `CONFIRM_EXPERIMENT_ID`, and rerun this cell. It validates the nine-run matrix and the audit gate before launching. Each completed window is checkpointed; after an interruption, rerun this cell to resume from the last saved window. The run order remains chronological for every seed.


In [ ]:
from temporal_ember_pilot.config import EXPERIMENT_ID, validate_run_matrix
from temporal_ember_pilot.runner import run_production_matrix

ENABLE_TRAINING = False
CONFIRM_EXPERIMENT_ID = ""
DEVICE = "cuda"

if not ENABLE_TRAINING:
    print("Training disabled. To launch the approved nine runs, explicitly set ENABLE_TRAINING=True and CONFIRM_EXPERIMENT_ID to the exact experiment ID.")
else:
    if CONFIRM_EXPERIMENT_ID != EXPERIMENT_ID:
        raise ValueError(f"Set CONFIRM_EXPERIMENT_ID exactly to {EXPERIMENT_ID}")
    if not globals().get("AUDIT_READY", False):
        raise RuntimeError("Run the CPU Audit step successfully before training")
    configs = validate_run_matrix()
    assert len(configs) == 9
    assert {config.seed for config in configs} == {42, 43, 44}
    assert {config.arm for config in configs} == {"full_hybrid", "anchor_only", "current_only"}
    run_status = run_production_matrix(
        archive_path=ARCHIVE_PATH,
        manifest_path=FROZEN_MANIFEST_PATH,
        output_root=OUTPUT_ROOT,
        feature_store_path=FEATURE_STORE_PATH,
        enable_training=ENABLE_TRAINING,
        confirmed_experiment_id=CONFIRM_EXPERIMENT_ID,
        device=DEVICE,
    )
    print(json.dumps(run_status, indent=2))


## 4. Report

Run this after all nine configurations finish. The exporter checks the completed run matrix, frozen schedule, per-window scoring IDs and counts, and acquisition support against the manifest. It reports January post-fit separately. The primary result gives each of the five post-January windows equal weight and includes all three paired seed differences. Negative or mixed differences are preserved.


In [ ]:
from temporal_ember_pilot.report import export_report
from temporal_ember_pilot.data import load_manifest

try:
    report = export_report(load_manifest(FROZEN_MANIFEST_PATH), OUTPUT_ROOT, REPORT_DIR)
except FileNotFoundError as error:
    print(f"Report pending: {error}. Run/Resume until all nine runs complete.")
else:
    print(json.dumps(report["primary_result"], indent=2))
    print("Per-run windows, sample counts, retention, acquisition, planned/realized replay, and GAN timing:")
    print(REPORT_DIR / "temporal_report.json")
    print(REPORT_DIR / "window_metrics.csv")
    print(REPORT_DIR / "new_family_acquisition.csv")
    print(REPORT_DIR / "frozen_split_manifest.json.gz")
    for name in ("temporal_report.json", "window_metrics.csv", "new_family_acquisition.csv", "frozen_split_manifest.json.gz"):
        print(f"{name} SHA-256: {sha256_file(REPORT_DIR / name)}")
